# 5f Draw a Digit

Turn a hand-drawn input into the same pixel contract as a classifier and see how distribution shift affects confidence.

**Budget:** 15–25 minutes; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), select a Python 3.10+ kernel, and use the dependency setup below if its packages are missing. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


## Install the notebook dependencies if needed

The next cell installs the shared CPU packages into **this notebook's selected Python kernel** using `%pip`. It includes **scikit-learn**, whose import name is `sklearn`, plus NumPy, pandas, matplotlib, Pillow, and IPython.

If the packages are missing or this is a fresh environment, set `INSTALL_PACKAGES = True` and run the cell once. Only do this when installation is permitted and pip is configured to use your approved package source. If the environment is already prepared, leave it `False`. The cell finds `requirements.txt` whether the kernel starts in the repository root or `05_Jupyter`.

After a successful installation, change the flag back to `False`, use **Kernel → Restart Kernel**, and run the notebook from the top. A restart clears old imports and variables. If installation fails or the source is blocked, stop here and ask the administrator to prepare the environment or approved mirror; do not bypass TLS checks.

PyTorch/CUDA, interactive widgets, ClearML, and live gateway extras remain optional. See the [support guide](README.md#prepare-one-approved-python-environment) before enabling them. Package installation can use the network; the lesson's default dataset path remains offline.


In [ ]:
from pathlib import Path
import sys

# Change to True only when this kernel needs the section's packages installed.
INSTALL_PACKAGES = False

SETUP_HERE = Path.cwd().resolve()
SETUP_LAB = SETUP_HERE if (SETUP_HERE / "image_lab.py").is_file() else SETUP_HERE / "05_Jupyter"
REQUIREMENTS = SETUP_LAB / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise FileNotFoundError("Start this kernel in the repository root or 05_Jupyter folder with requirements.txt present.")
print("Selected kernel Python:", sys.executable)

if INSTALL_PACKAGES:
    from IPython import get_ipython

    shell = get_ipython()
    if shell is None:
        raise RuntimeError("Run this setup cell in the notebook's IPython Python kernel.")
    # Equivalent to %pip install -r "{REQUIREMENTS}"; targets the selected kernel.
    shell.run_line_magic("pip", f'install -r "{REQUIREMENTS}"')
    print("Check pip output for errors. After success, set INSTALL_PACKAGES = False, restart the kernel, and run from the top.")
else:
    print("Installation skipped. If imports fail, set INSTALL_PACKAGES = True and rerun this cell first.")


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


In [ ]:
from image_lab import preprocess_drawing, predict_drawing
# A small standalone model; this notebook does not require a checkpoint from 5c.
data = load_image_data("digits", limit_train=1200, limit_test=300, seed=SEED)
model, metrics = train_baseline(data, C=1.0, max_iter=100, seed=SEED)
image_shape = data.train_images.shape[1:]
print("Model input:", data.name, image_shape, "Held-out accuracy:", metrics["accuracy"])


## Always-available NumPy fallback

This invented 8×8 pixel drawing approximates a zero. Edit the numbers below (0 = black background, 1 = bright stroke) and rerun. It is not a test-set example and has no guaranteed prediction. The preprocessing helper converts a local image to grayscale, resizes it to the training shape, and scales to [0,1]. It does not magically reproduce the centering and stroke statistics of a curated handwriting dataset.


In [ ]:
drawing = np.array([
    [0,0,0,1,1,0,0,0], [0,0,1,1,1,1,0,0],
    [0,1,1,0,0,1,1,0], [0,1,1,0,0,1,1,0],
    [0,1,1,0,0,1,1,0], [0,1,1,0,0,1,1,0],
    [0,0,1,1,1,1,0,0], [0,0,0,1,1,0,0,0]], dtype=np.float32)
def classify_drawing(pixels):
    prepared = preprocess_drawing(pixels, size=image_shape, invert=False)
    probabilities = model.predict_proba(prepared.reshape(1, -1))[0]
    ranking = np.argsort(probabilities)[-3:][::-1]
    plt.imshow(prepared, cmap="gray", vmin=0, vmax=1); plt.axis("off"); plt.show()
    print("Top guesses:", [(data.class_names[int(model.classes_[i])], round(float(probabilities[i]), 3))
                           for i in ranking])
    print("These probabilities are not a calibrated guarantee for hand-drawn inputs.")
classify_drawing(drawing)


## Optional interactive pixel drawing

If `ipywidgets` is installed and enabled in your JupyterLab frontend, click squares to paint a bright stroke, then **Predict**. This local 8×8 grid avoids a public web server or external upload. The first cell above remains the fallback for terminals, headless execution, or a frontend that cannot render widgets.


In [ ]:
SHOW_WIDGET = True
if SHOW_WIDGET:
    try:
        import ipywidgets as widgets
        from IPython.display import display
        pixels = [widgets.ToggleButton(value=False, layout=widgets.Layout(width="27px", height="27px"))
                  for _ in range(64)]
        grid = widgets.GridBox(pixels, layout=widgets.Layout(grid_template_columns="repeat(8, 27px)", grid_gap="1px"))
        predict_button = widgets.Button(description="Predict")
        clear_button = widgets.Button(description="Clear")
        result_box = widgets.Output()
        def on_predict(_):
            with result_box:
                result_box.clear_output(wait=True)
                classify_drawing(np.array([p.value for p in pixels], dtype=np.float32).reshape(8, 8))
        def on_clear(_):
            for pixel in pixels:
                pixel.value = False
            result_box.clear_output()
        predict_button.on_click(on_predict); clear_button.on_click(on_clear)
        display(widgets.VBox([grid, widgets.HBox([predict_button, clear_button]), result_box]))
    except ImportError:
        print("ipywidgets unavailable; use the editable NumPy drawing above.")


## Optional local PNG

To use your own local file, inspect it first and load it with Pillow: `Image.open(path).convert("L")`, then pass it to `preprocess_drawing`. Choose inversion explicitly if the input is dark ink on a white background. Do not load files from untrusted origins or upload personal handwriting to a remote service for this exercise. An all-black or all-white input is not a meaningful digit even if a classifier returns a class.


## Checkpoint, exercise, and cleanup

**Checkpoint:** Both the editable array and the optional widget use the same preprocessing/model. You can explain why confidence is not correctness.

**Try it:** Draw the same digit centered, shifted, thin, and thick. Record the top guesses and explain why a model that does well on curated test images can fail on your drawing.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5g: Test Data Quality and Robustness](05g_Test_Data_Quality_and_Robustness.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
